# MankiwEcoLab Interactive Lab

An interactive tour of the economic models implemented in this repository.

- **Consumer choice theory** - budget constraints, Cobb-Douglas utility, optimal bundles
- **Game theory** - Nash equilibrium, dominant strategies, mixed strategies, Cournot oligopoly
- **Loanable funds market** - equilibrium interest rates, fiscal policy, crowding out
- **IS-LM model** - simultaneous goods and money market equilibrium

Run each cell in order. All cells are self-contained and deterministic.

## 1. Consumer Choice Theory

A consumer maximises utility `U(x,y) = x^a * y^(1-a)` subject to the budget
constraint `Px*x + Py*y = I`. The optimum satisfies the tangency condition
`MRS = Px/Py`.

In [ ]:
from micro import BudgetConstraint, CobbDouglasUtility, ConsumerChoice

budget = BudgetConstraint(income=1000, price_x=10, price_y=20)
utility = CobbDouglasUtility(alpha=0.5)
choice = ConsumerChoice(budget, utility)

print(f"Budget line: 10x + 20y = 1000")
print(f"Optimal bundle: x*={choice.optimal_x:.2f}, y*={choice.optimal_y:.2f}")
print(f"Tangency condition holds: {choice.verify_tangency()}")
print(f"Budget satisfied: {choice.verify_budget_satisfied()}")

### Try it yourself

Change the parameters below and re-run the cell. Observe how the optimal bundle
responds to a change in income or price (the law of demand).

In [ ]:
# --- Experiment with these values ---
INCOME = 1000
PRICE_X = 10
PRICE_Y = 20
ALPHA = 0.5
# ------------------------------------

b = BudgetConstraint(income=INCOME, price_x=PRICE_X, price_y=PRICE_Y)
u = CobbDouglasUtility(alpha=ALPHA)
c = ConsumerChoice(b, u)
bundle = c.optimal_bundle()
print(f"Optimal: x*={bundle['x']:.2f}, y*={bundle['y']:.2f}, utility={bundle['utility']:.2f}")
print(f"Expenditure = {bundle['expenditure']:.2f}")

## 2. Game Theory

The Prisoner's Dilemma shows how rational self-interest can prevent mutually
beneficial cooperation. The Matching Pennies game has only a mixed-strategy
Nash equilibrium.

In [ ]:
from micro import prisoners_dilemma, matching_pennies, CournotGame

pd = prisoners_dilemma()
print("Prisoner's Dilemma:")
print(f"  Dominant strategies: A={pd.dominant_strategies()['A']}, B={pd.dominant_strategies()['B']}")
print(f"  Pure Nash equilibrium: {pd.pure_nash_equilibria()[0]['A_strategy']} / {pd.pure_nash_equilibria()[0]['B_strategy']}")

mp = matching_pennies()
mixed = mp.mixed_strategy_equilibrium()
print(f"\nMatching Pennies mixed equilibrium: p={mixed['p']:.2f}, q={mixed['q']:.2f}")

In [ ]:
cg = CournotGame(num_firms=2, demand_intercept=100, demand_slope=1, marginal_cost=20)
print(f"Market demand: P = 100 - Q, MC = 20")
nash = cg.nash_equilibrium()
collusion = cg.collusion_output()
comp = cg.competitive_output()
print(f"Nash:    q={nash['per_firm_output']:.2f}, P={nash['price']:.2f}")
print(f"Cartel:  Q={collusion['total_output']:.2f}, P={collusion['price']:.2f}")
print(f"Perfect: Q={comp['total_output']:.2f}, P={comp['price']:.2f}")
print(f"\nThe Nash price lies between monopoly and perfect competition.")

## 3. Loanable Funds Market

The market for loanable funds matches savings (supply) with investment
(demand). Government borrowing shifts demand right and raises the interest
rate, partially *crowding out* private investment.

In [ ]:
from macro import LoanableFundsModel

lf = LoanableFundsModel(
    savings_autonomous=800, savings_sensitivity=200,
    investment_autonomous=1200, investment_sensitivity=400,
    government_borrowing=0,
)
print(f"Equilibrium interest rate: {lf.equilibrium_rate():.2%}")

result = lf.with_fiscal_policy(additional_borrowing=200)
print(f"After government borrows 200:")
print(f"  Rate change: {result['interest_rate_change']:.2%}")
print(f"  Private investment change: {result['investment_change']:.2f}")
print(f"  Crowding out: {result['crowding_out']:.2f}")

## 4. IS-LM Model

The IS-LM model describes the simultaneous equilibrium of the goods market
(IS curve) and the money market (LM curve). The intersection determines
output `Y` and the interest rate `r`.

In [ ]:
from macro import ISLMModel

islm = ISLMModel(
    consumption_autonomous=100, marginal_propensity_to_consume=0.8,
    tax_rate=0.25, investment_autonomous=200, investment_sensitivity=1000,
    government_spending=300, real_money_supply=500,
    money_demand_income=0.5, money_demand_interest=200,
)
eq = islm.equilibrium()
print(f"Equilibrium: Y={eq['output']:.2f}, r={eq['interest_rate']:.2%}")
print(f"On both curves: {islm.verify_on_curves()}")

fp = islm.fiscal_policy(spending_change=50)
mp = islm.monetary_policy(money_supply_change=100)
print(f"\nExpansionary fiscal (+50 G):  dY={fp['output_change']:.2f}, dr={fp['interest_rate_change']:.2%}")
print(f"Expansionary monetary (+100 M): dY={mp['output_change']:.2f}, dr={mp['interest_rate_change']:.2%}")

## 5. Market Simulation

The core agent-based market simulation matches consumers and producers over
successive rounds until prices converge to equilibrium.

In [ ]:
from market import Market
from utils.economics import create_agents

consumer_params = {
    'income_mean': 1000, 'income_std': 200, 'income_min': 500,
    'alpha_mean': 100, 'alpha_std': 10, 'beta_mean': 0.5, 'beta_std': 0.05,
}
producer_params = {
    'fixed_cost_mean': 300, 'fixed_cost_std': 50, 'mc_a_mean': 10,
    'mc_a_std': 2, 'mc_b_mean': 0.3, 'mc_b_std': 0.05,
    'max_capacity_mean': 100, 'max_capacity_std': 20,
}

consumers, producers = create_agents(
    300, 80, consumer_params, producer_params, random_seed=42)
market = Market(consumers, producers, initial_price=50,
                price_adjustment_speed=0.1)
for _ in range(40):
    market.run_round()

print(f"Rounds run: {len(market.price_history)}")
print(f"Final price: {market.current_price:.2f}")
print(f"Equilibrium reached: {market.equilibrium_reached}")
stats = market.get_market_stats()
print(f"Total surplus: {stats['total_surplus']:.2f}")

### Next steps

- Run the full CLI demos: `python main.py --demo`, `python main.py --macro`, `python main.py --experiments`
- Explore all experiments interactively in `experiments.py`
- Read the model derivations in `docs/models.md`